In [1]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [2]:
FEATURE_DIR = Path("artifacts/features")
train = pd.read_csv("artifacts/train.csv")
validation = pd.read_csv("artifacts/validation.csv")
test = pd.read_csv("artifacts/test.csv")

selected_features_df = pd.read_csv("artifacts/eda/selected_features.csv")
selected_features = selected_features_df["feature"].tolist()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)
print("Selected features:", selected_features)


Train: (67533, 22)
Validation: (14471, 22)
Test: (14472, 22)
Selected features: ['order_purchase_timestamp', 'order_approved_at', 'order_estimated_delivery_date', 'item_count', 'unique_products', 'unique_sellers', 'total_price', 'total_freight', 'avg_item_price', 'payment_count', 'total_payment_value', 'max_installments', 'customer_zip_code_prefix', 'customer_state']


## 1. Prediction-time feature set

These are the features selected in Notebook 4:

- `order_purchase_timestamp`
- `order_approved_at`
- `order_estimated_delivery_date`
- `item_count`
- `unique_products`
- `unique_sellers`
- `total_price`
- `total_freight`
- `avg_item_price`
- `payment_count`
- `total_payment_value`
- `max_installments`
- `customer_zip_code_prefix`
- `customer_state`

Post-delivery columns such as `order_delivered_customer_date` and the EDA-only `delivery_days` are excluded to avoid future leakage.

**Prediction-time assumption:** prediction happens after order approval and when the estimated delivery date is available, so `order_approved_at` is usable.


In [3]:
for name, df in {"train": train, "validation": validation, "test": test}.items():
    missing = [c for c in selected_features if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing selected columns: {missing}")

forbidden_future_columns = [
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "delivery_days"]
leakage = [c for c in selected_features if c in forbidden_future_columns]
if leakage:
    raise ValueError(f"Future/leakage columns selected: {leakage}")

print("Feature availability and leakage checks passed.")


Feature availability and leakage checks passed.


## 2. Date/time feature engineering

Raw datetime columns are converted into model-friendly features:

- purchase year/month/day-of-week/hour
- approval delay in hours
- estimated delivery horizon in days

All of these use information available at prediction time. The actual delivery dates are never used.


In [4]:
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_estimated_delivery_date"]

def engineer_features(df):
    data = df[selected_features].copy()

    for col in DATE_COLUMNS:
        data[col] = pd.to_datetime(data[col], errors="coerce")

    purchase = data["order_purchase_timestamp"]
    approved = data["order_approved_at"]
    estimated = data["order_estimated_delivery_date"]

    data["purchase_year"] = purchase.dt.year
    data["purchase_month"] = purchase.dt.month
    data["purchase_day_of_week"] = purchase.dt.dayofweek
    data["purchase_hour"] = purchase.dt.hour
    data["approval_delay_hours"] = (
        (approved - purchase).dt.total_seconds() / 3600)
    data["estimated_delivery_days"] = (
        (estimated - approved).dt.total_seconds() / (24 * 3600))

    return data.drop(columns=DATE_COLUMNS)


In [5]:
X_train_raw = engineer_features(train)
X_validation_raw = engineer_features(validation)
X_test_raw = engineer_features(test)

y_train = train["is_late"].copy()
y_validation = validation["is_late"].copy()
y_test = test["is_late"].copy()

print("Engineered shapes:")
print(X_train_raw.shape, X_validation_raw.shape, X_test_raw.shape)
print("\nEngineered columns:")
print(X_train_raw.columns.tolist())


Engineered shapes:
(67533, 17) (14471, 17) (14472, 17)

Engineered columns:
['item_count', 'unique_products', 'unique_sellers', 'total_price', 'total_freight', 'avg_item_price', 'payment_count', 'total_payment_value', 'max_installments', 'customer_zip_code_prefix', 'customer_state', 'purchase_year', 'purchase_month', 'purchase_day_of_week', 'purchase_hour', 'approval_delay_hours', 'estimated_delivery_days']


In [6]:
missing_before = pd.DataFrame({
    "missing_count": X_train_raw.isna().sum(),
    "missing_percentage": X_train_raw.isna().mean() * 100})
missing_before[missing_before["missing_count"] > 0].sort_values(
    "missing_percentage", ascending=False)


,missing_count,missing_percentage
estimated_delivery_days,14,0.020731
approval_delay_hours,14,0.020731
payment_count,1,0.001481
max_installments,1,0.001481
total_payment_value,1,0.001481


## 3. Imputation, encoding, and scaling

Numerical features use **median imputation + StandardScaler**.

Categorical features use **most-frequent imputation + OneHotEncoder(handle_unknown="ignore")**.

`handle_unknown="ignore"` allows production data to contain a category that was not seen during training.

The scaler is included because the preprocessing is model-ready for models such as logistic regression; tree-based models do not strictly require scaling.


In [7]:
numeric_features = X_train_raw.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train_raw.select_dtypes(include=["object", "category"]
).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)


Numeric: ['item_count', 'unique_products', 'unique_sellers', 'total_price', 'total_freight', 'avg_item_price', 'payment_count', 'total_payment_value', 'max_installments', 'customer_zip_code_prefix', 'purchase_year', 'purchase_month', 'purchase_day_of_week', 'purchase_hour', 'approval_delay_hours', 'estimated_delivery_days']
Categorical: ['customer_state']


C:\Users\rosto\AppData\Local\Temp\ipykernel_4800\4088782861.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train_raw.select_dtypes(include=["object", "category"]


In [8]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)], remainder="drop")

print("Preprocessor created.")


Preprocessor created.


## 4. Fit on training only

This is the anti-leakage step.

The medians, scaling parameters, category vocabulary, and one-hot mapping are learned only from `X_train_raw`. Validation and test are transformed using those fitted objects without calling `fit()` again.


In [9]:
X_train_transformed = preprocessor.fit_transform(X_train_raw)

X_validation_transformed = preprocessor.transform(X_validation_raw)
X_test_transformed = preprocessor.transform(X_test_raw)

print("Train transformed:", X_train_transformed.shape)
print("Validation transformed:", X_validation_transformed.shape)
print("Test transformed:", X_test_transformed.shape)


Train transformed: (67533, 43)
Validation transformed: (14471, 43)
Test transformed: (14472, 43)


In [10]:
feature_names = preprocessor.get_feature_names_out()

X_train_final = pd.DataFrame(
    X_train_transformed, columns=feature_names, index=train.index)
X_validation_final = pd.DataFrame(
    X_validation_transformed, columns=feature_names, index=validation.index)
X_test_final = pd.DataFrame(
    X_test_transformed, columns=feature_names, index=test.index)

print("Final feature count:", len(feature_names))
print("First 20 features:", feature_names[:20])


Final feature count: 43
First 20 features: ['numeric__item_count' 'numeric__unique_products'
 'numeric__unique_sellers' 'numeric__total_price' 'numeric__total_freight'
 'numeric__avg_item_price' 'numeric__payment_count'
 'numeric__total_payment_value' 'numeric__max_installments'
 'numeric__customer_zip_code_prefix' 'numeric__purchase_year'
 'numeric__purchase_month' 'numeric__purchase_day_of_week'
 'numeric__purchase_hour' 'numeric__approval_delay_hours'
 'numeric__estimated_delivery_days' 'categorical__customer_state_AC'
 'categorical__customer_state_AL' 'categorical__customer_state_AM'
 'categorical__customer_state_AP']


## 5. Verification

In [11]:
assert list(X_train_final.columns) == list(X_validation_final.columns)
assert list(X_train_final.columns) == list(X_test_final.columns)
assert "is_late" not in X_train_final.columns
assert len(X_train_final) == len(train)
assert len(X_validation_final) == len(validation)
assert len(X_test_final) == len(test)
assert not X_train_final.isna().any().any()
assert not X_validation_final.isna().any().any()
assert not X_test_final.isna().any().any()

print("All verification checks passed: same feature order, no missing values, target excluded.")


All verification checks passed: same feature order, no missing values, target excluded.


In [12]:
from pathlib import Path

FEATURE_DIR = Path("artifacts/features")
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print(FEATURE_DIR.resolve())
print(FEATURE_DIR.exists())

D:\لاب\samar\MLops\تاسكات\olist_ml_pipeline task2\artifacts\features
True


In [13]:
train_features = X_train_final.copy()
train_features["is_late"] = y_train.to_numpy()

validation_features = X_validation_final.copy()
validation_features["is_late"] = y_validation.to_numpy()

test_features = X_test_final.copy()
test_features["is_late"] = y_test.to_numpy()

train_features.to_csv(FEATURE_DIR / "train_features.csv", index=False)
validation_features.to_csv(FEATURE_DIR / "validation_features.csv", index=False)
test_features.to_csv(FEATURE_DIR / "test_features.csv", index=False)

print("Final feature tables saved.")


Final feature tables saved.


In [14]:
preprocessor_path = FEATURE_DIR / "preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path)

feature_list_path = FEATURE_DIR / "feature_list.csv"
pd.DataFrame({"feature": feature_names}).to_csv(feature_list_path, index=False)

metadata = {
    "selected_raw_features": selected_features,
    "engineered_numeric_features": numeric_features,
    "engineered_categorical_features": categorical_features,
    "final_feature_count": int(len(feature_names)),
    "numeric_imputation": "median",
    "categorical_imputation": "most_frequent",
    "categorical_encoding": "OneHotEncoder(handle_unknown='ignore')",
    "scaling": "StandardScaler",
    "fit_data": "training split only",
    "future_columns_excluded": forbidden_future_columns}
with open(FEATURE_DIR / "preprocessing_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved:", preprocessor_path)
print("Saved:", feature_list_path)
print("Saved preprocessing metadata.")


Saved: artifacts\features\preprocessor.joblib
Saved: artifacts\features\feature_list.csv
Saved preprocessing metadata.


## 6. Production inference pattern

Production must load the saved preprocessing pipeline and call `transform()`.

**Do not call `.fit()` or `.fit_transform()` on new data.**

Flow:

`new raw order → same feature engineering → saved preprocessor.transform() → model.predict()`


In [15]:
loaded_preprocessor = joblib.load(FEATURE_DIR / "preprocessor.joblib")

# Demonstration using one existing row as if it were a new order.
new_order_raw = train.iloc[[0]].copy()
new_order_engineered = engineer_features(new_order_raw)
new_order_ready = loaded_preprocessor.transform(new_order_engineered)

print("Production-style transformed shape:", new_order_ready.shape)
print("No fitting was performed on the new row.")


Production-style transformed shape: (1, 43)
No fitting was performed on the new row.


## 7. Final artifacts

Created under `artifacts/features/`:

- `train_features.csv`
- `validation_features.csv`
- `test_features.csv`
- `preprocessor.joblib` — fitted imputers + encoder + scaler
- `feature_list.csv` — exact final feature order
- `preprocessing_metadata.json`


In [16]:
print("Saved feature-engineering artifacts:")
for file in sorted(FEATURE_DIR.iterdir()):
    print(" -", file.name)


Saved feature-engineering artifacts:
 - feature_list.csv
 - preprocessing_metadata.json
 - preprocessor.joblib
 - test_features.csv
 - train_features.csv
 - validation_features.csv
